In [3]:
!pip install rrcf

In [8]:
import numpy as np
import pandas as pd
import rrcf

def generate_synthetic_vitals(seed=42):
    np.random.seed(seed)
    days = 30
    n_users = 50
    total_records = days * n_users

    heart_rate = np.random.normal(75, 10, total_records)
    systolic_bp = np.random.normal(120, 15, total_records)
    blood_sugar = np.random.normal(100, 20, total_records)
    steps = np.random.poisson(8000, total_records)
    water_intake = np.random.normal(2.5, 0.5, total_records)

    df = pd.DataFrame({
        'heart_rate': heart_rate,
        'systolic_bp': systolic_bp,
        'blood_sugar': blood_sugar,
        'steps': steps,
        'water_intake': water_intake
    })

    anomaly_indices = np.random.choice(total_records, size=int(total_records * 0.05), replace=False)
    for idx in anomaly_indices:
        df.loc[idx, 'heart_rate'] += np.random.uniform(40, 60)
        df.loc[idx, 'systolic_bp'] -= np.random.uniform(30, 50)
        df.loc[idx, 'water_intake'] -= np.random.uniform(1.0, 2.0)

    return df, anomaly_indices

def main():
    training_file = "realistic_synthetic_health_vitals.csv"
    features = ['heart_rate', 'systolic_bp', 'blood_sugar', 'steps', 'water_intake']

    num_trees = 100
    tree_size = 256
    forest = []

    for _ in range(num_trees):
        tree = rrcf.RCTree()
        forest.append(tree)

    if training_file:
        print(f"Loading and training on: {training_file}")
        try:
            df_train = pd.read_csv(training_file)

            train_data = df_train[features].values

            for i, point in enumerate(train_data):
                for tree in forest:
                    if len(tree.leaves) > tree_size:
                        tree.forget_point(i - tree_size)
                    tree.insert_point(point, index=i)
            print("Global RCF training complete.")
        except FileNotFoundError:
            print(f"Error: Could not find {training_file}. Proceeding without pre-training.")
        except KeyError as e:
             print(f"Error: Your training file is missing a required feature column: {e}")
             return
    else:
        print("No training file provided. Initializing an empty RCF model.")

    print("\nGenerating synthetic clinical data (Seed: 42)...")
    df_test, true_anomalies = generate_synthetic_vitals(seed=42)

    test_data = df_test[features].values

    print("Scoring synthetic dataset...")
    anomaly_scores = np.zeros(len(test_data))

    idx_offset = 1000000

    for i, point in enumerate(test_data):
        score = 0
        current_idx = i + idx_offset

        for tree in forest:
            if len(tree.leaves) == 0:
                tree.insert_point(point, index=current_idx)
                continue

            tree.insert_point(point, index=current_idx)
            score += tree.codisp(current_idx)
            tree.forget_point(current_idx)

        anomaly_scores[i] = score / num_trees

    df_test['anomaly_score'] = anomaly_scores
    df_test['is_true_anomaly'] = df_test.index.isin(true_anomalies)

    print("\n--- Testing Complete ---")
    print("\nTop 10 Data Points with the Highest Anomaly Scores:")

    top_anomalies = df_test.sort_values('anomaly_score', ascending=False).head(10)
    print(top_anomalies.to_string())

if __name__ == "__main__":
    main()

Loading and training on: realistic_synthetic_health_vitals.csv
Global RCF training complete.

Generating synthetic clinical data (Seed: 42)...
Scoring synthetic dataset...

--- Testing Complete ---

Top 10 Data Points with the Highest Anomaly Scores:
      heart_rate  systolic_bp  blood_sugar  steps  water_intake  anomaly_score  is_true_anomaly
0      79.967142   131.675416    61.843849   7946      2.462827      27.754707            False
1      73.617357   111.732214    82.792300   8016      2.407724      17.193675            False
1012   95.752609   128.233258    80.300137   8131      1.776751      17.149150            False
614    99.457520   131.112365   113.857115   7983      2.692855      17.049059            False
59    137.093167    75.753102    92.086386   7919     -0.504118      17.035665             True
44    104.351601    68.496663   114.292196   7834      1.829589      17.020672             True
1043  131.503028    76.306763   105.992954   7989      0.381477      16.97303